In [1]:
import os
import sys
import tqdm

project_root = os.path.abspath(os.path.join(os.getcwd(), '../..'))
sys.path.append(project_root)

from model import *
from dataset_loader import *

model = SpeechT5(encoder_type='wav2vec')

/media/zawiatgf/New Volume/Personal Files/Abdurrahman Zawia/University/Grad Project/Speech-To-Speech-Model/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loading SpeechT5 components (encoder_type='wav2vec')...
Using device: cuda


In [ ]:

model.fine_tune(source_lang="en", target_lang="de", batch_size=1, epochs=8, learning_rate=1e-5)
model.save("speecht5_en_de_wav2vec")

Starting fine-tuning: en -> de  [encoder_type='wav2vec']
Loading preprocessed data from /media/zawiatgf/New Volume/Personal Files/Abdurrahman Zawia/University/Grad Project/Speech-To-Speech-Model/datasets/processed_speecht5_wav2vec_en_de_v1...
Initializing X-Vector classifier for embedding extraction...


/media/zawiatgf/New Volume/Personal Files/Abdurrahman Zawia/University/Grad Project/Speech-To-Speech-Model/.venv/lib/python3.12/site-packages/speechbrain/utils/autocast.py:188: FutureWarning: `torch.cuda.amp.custom_fwd(args...)` is deprecated. Please use `torch.amp.custom_fwd(args..., device_type='cuda')` instead.
  wrapped_fwd = torch.cuda.amp.custom_fwd(fwd, cast_inputs=cast_inputs)


Extracting target speaker embedding...
Streaming 1 sample from google/fleurs for de...
Embedding extracted successfully.
Cleaning up speaker classifier to free VRAM...
Freezing Feature Encoder.


Epoch 1/8:   0%|          | 0/13438 [00:00<?, ?it/s]

In [ ]:

model.save('speecht5_en_de_partially_trained')

In [ ]:
audio_data = load_data(
    dataset=['fleurs'],
    encoding=None,
    lang=['en', 'de'],
    num_samples=2000,
    split='train'
)


In [ ]:

sample_id = 3
play_audio(audio_data['en'][sample_id])
# audio_data['en'][sample_id]


In [ ]:

play_audio(audio_data['de'][sample_id])
# audio_data['de'][sample_id]

In [ ]:
predicted_audio = model.run_inference(
    audio_data['de'][sample_id]['audio']['array'], 
    audio_data['de'][sample_id]['audio']['sampling_rate']
    )
play_audio(predicted_audio)

In [ ]:
from dataset_loader import save_audio

save_audio(
    audio_data['en'][sample_id],
    "input.wav"
    )

save_audio(
    audio_data['de'][sample_id],
    "expected.wav"
    )

save_audio(
    predicted_audio,
    "output.wav"
    )

In [ ]:

model.fine_tune(source_lang="en", target_lang="de", batch_size=1, epochs=10, learning_rate=1e-5)
model.save("speecht5_en_de")

In [ ]:
import os
import sys

project_root = os.path.abspath(os.path.join(os.getcwd(), '../..'))
sys.path.append(project_root)

from model import *
from dataset_loader import load_data, play_audio

sample_data = load_data(
    start_idx=0,
    num_samples=10,
    lang=['en', 'de'],
    split="train",
    dataset=["seamless_align"],
    encoding=None
)

model = SpeechT5()
model.load("speecht5_en_de_interrupted")

In [ ]:
de_predicted = model.run_inference(
    sample_data['en'][1]['audio']['array'],
    sample_data['en'][1]['audio']['sampling_rate']
    )

play_audio(de_predicted)

In [ ]:
play_audio(sample_data['en'][1])

In [ ]:
import soundfile as sf

sf.write(
    'de_predicted.wav', 
    # sample_data['de'][1]['audio']['array'], 
    # sample_data['de'][1]['audio']['sampling_rate']
    de_predicted['audio']['array'],
    de_predicted['audio']['sampling_rate']
    )

print(f"Audio saved")